In [0]:
# Import configuration from config.py
exec(open('../config.py').read())

print(f"Config loaded: catalog={catalog}, schema={schema}")
print(f"App: {app_name}, Lakebase: {lakebase_instance_name}, SA: {sa_name}")

In [0]:
import os
import sys
import json
import time
import logging
from databricks.sdk import WorkspaceClient

# Load helpers
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from resources.brick_setup_functions import AgentBricksManager

# Set DATABRICKS_HOST from notebook context for WorkspaceClient()
from databricks.sdk.runtime import dbutils
notebook_context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host = notebook_context.browserHostName().get()
os.environ["DATABRICKS_HOST"] = f"https://{host}"

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)
logger = logging.getLogger(__name__)

w = WorkspaceClient()
brick_manager = AgentBricksManager(w)

print(f"Workspace: https://{host}")
print(f"Supervisor Agent: {sa_name}")
print(f"App Name: {app_name}")
print(f"Lakebase Instance: {lakebase_instance_name}")

# Deploy Chatbot App with Supervisor Agent & Lakebase

This notebook:
1. Discovers the MAS serving endpoint, Genie spaces, and UC functions
2. Creates a Lakebase database instance for chat history
3. Creates the Databricks App with serving endpoint resources
4. Grants the app service principal permissions on Genie spaces, UC functions, and data tables
5. Deploys source code to the app

In [0]:
# Step 1: Discover the MAS tile and serving endpoint name
print(f"🔍 Looking up Supervisor Agent '{sa_name}'...")

# Search for MAS tile type (find_by_name only searches KA tiles)
resp = w.api_client.do(
    'GET',
    f'/api/2.0/tiles?filter=name_contains%3D{sa_name}%26%26tile_type%3DMAS'
)
tiles = [t for t in resp.get('tiles', []) if t.get('name') == sa_name]
assert tiles, f"Supervisor Agent '{sa_name}' not found. Run 04_instructor_setup_sa first."

mas_tile_id = tiles[0]['tile_id']
mas_data = brick_manager.mas_get(mas_tile_id)
mas_info = mas_data['multi_agent_supervisor']

# Try multiple possible locations for the endpoint name
mas_endpoint_name = (
    mas_info.get('endpoint_name') or
    mas_info.get('status', {}).get('endpoint_name') or
    mas_info.get('tile', {}).get('endpoint_name')
)

# If not found in MAS response, search serving endpoints by naming convention
if not mas_endpoint_name:
    print("   Endpoint name not in MAS response, searching serving endpoints...")
    endpoints = list(w.serving_endpoints.list())
    for ep in endpoints:
        ep_name = ep.name
        if 'mas-' in ep_name or mas_tile_id[:8] in ep_name:
            mas_endpoint_name = ep_name
            print(f"   Found matching endpoint: {ep_name}")
            break

if not mas_endpoint_name:
    print("\n⚠️ Could not auto-discover MAS endpoint name.")
    print("Available serving endpoints:")
    for ep in endpoints:
        print(f"   - {ep.name} (state: {ep.state})")
    raise Exception("Could not find MAS serving endpoint. Set mas_endpoint_name manually.")

mas_status = mas_info.get('status', {}).get('endpoint_status')

print(f"\n✅ Found MAS: {sa_name}")
print(f"   Tile ID: {mas_tile_id}")
print(f"   Endpoint: {mas_endpoint_name}")
print(f"   Status: {mas_status}")

if mas_status != 'ONLINE':
    print(f"\n⚠️ MAS is not ONLINE yet (status: {mas_status}).")
    print("The app will be deployed but may not work until the MAS is ready.")

# Collect all sub-agent info for permission grants
sub_agent_endpoints = []
genie_space_ids = []
uc_function_paths = []

agents = mas_info.get('agents', [])
print(f"\n📋 Sub-agents ({len(agents)}):")
for i, agent in enumerate(agents):
    agent_name = agent.get('name', 'Unknown')
    agent_type = agent.get('agent_type', 'Unknown')
    print(f"  {i+1}. {agent_name} ({agent_type})")

    # Serving endpoints (KA agents)
    if 'serving_endpoint' in agent:
        ep_name = agent['serving_endpoint']['name']
        sub_agent_endpoints.append(ep_name)
        print(f"      Endpoint: {ep_name}")

    # Genie spaces - agent_type can be "genie", "genie-space", or "genie_space"
    if 'genie_space' in agent:
        space_id = agent['genie_space']['id']
        genie_space_ids.append(space_id)
        print(f"      Genie Space ID: {space_id}")

    # UC functions - agent_type can be "function", "unity-catalog-function", or "unity_catalog_function"
    if 'unity_catalog_function' in agent:
        uc_path = agent['unity_catalog_function']['uc_path']
        full_path = f"{uc_path['catalog']}.{uc_path['schema']}.{uc_path['name']}"
        uc_function_paths.append(full_path)
        print(f"      UC Function: {full_path}")

print(f"\n🔑 Endpoints needing CAN_QUERY grants: {len(sub_agent_endpoints)}")
for ep in sub_agent_endpoints:
    print(f"   - {ep}")
print(f"\n🎯 Genie spaces needing permissions: {len(genie_space_ids)}")
for gid in genie_space_ids:
    print(f"   - {gid}")
print(f"\n🔧 UC functions needing EXECUTE grant: {len(uc_function_paths)}")
for uf in uc_function_paths:
    print(f"   - {uf}")

In [0]:
# Step 2: Prepare deployment configuration
# No need to modify databricks.yml - we'll deploy directly via REST API

print("📋 Deployment Configuration:")
print(f"   MAS Endpoint: {mas_endpoint_name}")
print(f"   App Name: {app_name}-{app_resource_suffix}")
print(f"   Lakebase Instance: {lakebase_instance_name}-{app_resource_suffix}")
print(f"   Sub-agent endpoints: {len(sub_agent_endpoints)}")

# Build the list of all serving endpoints that need CAN_QUERY
all_endpoints = [mas_endpoint_name] + sub_agent_endpoints
print(f"\n🔑 All endpoints for CAN_QUERY permission:")
for ep in all_endpoints:
    print(f"   - {ep}")

In [0]:
# Step 3: Create Lakebase database instance, wait for AVAILABLE, and ensure database exists
from databricks.sdk.errors import NotFound as _NotFound

lakebase_full_name = f"{lakebase_instance_name}-{app_resource_suffix}"
lakebase_db_name = "databricks_postgres"
print(f"Setting up Lakebase database instance: {lakebase_full_name}...")

# Check if it already exists
lakebase_exists = False
try:
    existing_db = w.api_client.do('GET', f'/api/2.0/database/instances/{lakebase_full_name}')
    lakebase_state = existing_db.get('state', 'unknown')
    print(f"Lakebase instance already exists: {lakebase_full_name} (state: {lakebase_state})")
    lakebase_exists = True

    # Resume if stopped
    if lakebase_state == 'STOPPED' or existing_db.get('effective_stopped'):
        print("   Instance is stopped, resuming...")
        w.api_client.do('PATCH', f'/api/2.0/database/instances/{lakebase_full_name}', body={
            'stopped': False
        })
        print("   Resume requested")
except _NotFound:
    pass
except Exception as e:
    if 'not found' in str(e).lower() or 'Resource not found' in str(e):
        pass
    else:
        print(f"Warning checking Lakebase: {e}")

# Create if it doesn't exist
if not lakebase_exists:
    print(f"   Creating new Lakebase instance...")
    try:
        result = w.api_client.do('POST', '/api/2.0/database/instances', body={
            'name': lakebase_full_name,
            'capacity': 'CU_1'
        })
        print(f"Lakebase instance created: {lakebase_full_name}")
    except Exception as create_err:
        if 'already exists' in str(create_err).lower():
            print(f"Lakebase instance already exists: {lakebase_full_name}")
        else:
            raise

# Wait for AVAILABLE state
print(f"\nWaiting for Lakebase to be AVAILABLE...")
for i in range(36):  # up to 3 minutes
    try:
        db_info = w.api_client.do('GET', f'/api/2.0/database/instances/{lakebase_full_name}')
        state = db_info.get('state', 'UNKNOWN')
        if state == 'AVAILABLE':
            print(f"Lakebase is AVAILABLE!")
            break
        print(f"   State: {state} ({(i+1)*5}s)")
    except Exception:
        print(f"   Checking... ({(i+1)*5}s)")
    time.sleep(5)
else:
    print("Warning: Lakebase may not be ready yet. Continuing anyway...")

# Ensure the database exists inside the instance
print(f"\nChecking for database '{lakebase_db_name}' in instance...")
try:
    databases = w.api_client.do('GET', f'/api/2.0/database/instances/{lakebase_full_name}/databases')
    db_names = [d.get('name') for d in databases.get('databases', [])]
    print(f"   Existing databases: {db_names}")
    if lakebase_db_name not in db_names:
        print(f"   Creating database '{lakebase_db_name}'...")
        w.api_client.do('POST', f'/api/2.0/database/instances/{lakebase_full_name}/databases', body={
            'name': lakebase_db_name
        })
        print(f"   Database '{lakebase_db_name}' created!")
    else:
        print(f"   Database '{lakebase_db_name}' already exists")
except Exception as e:
    # If listing fails, try creating directly (it may error if already exists, which is fine)
    print(f"   Could not list databases ({e}), attempting to create '{lakebase_db_name}'...")
    try:
        w.api_client.do('POST', f'/api/2.0/database/instances/{lakebase_full_name}/databases', body={
            'name': lakebase_db_name
        })
        print(f"   Database '{lakebase_db_name}' created!")
    except Exception as e2:
        if 'already exists' in str(e2).lower():
            print(f"   Database '{lakebase_db_name}' already exists")
        else:
            print(f"   Warning: Could not create database: {e2}")
            print(f"   The app deployment may handle this automatically.")

In [0]:
# Step 4: Create or update the Databricks App
from databricks.sdk.errors import NotFound
import requests as req

app_full_name = f"{app_name}-{app_resource_suffix}"
print(f"Creating/updating Databricks App: {app_full_name}...")

# Build app resources list
app_resources = [
    {
        "name": "serving-endpoint",
        "description": "Multi-Agent Supervisor serving endpoint",
        "serving_endpoint": {
            "name": mas_endpoint_name,
            "permission": "CAN_QUERY"
        }
    }
]

for i, ep_name in enumerate(sub_agent_endpoints):
    app_resources.append({
        "name": f"sub-agent-{i+1}",
        "description": f"Sub-agent endpoint: {ep_name}",
        "serving_endpoint": {
            "name": ep_name,
            "permission": "CAN_QUERY"
        }
    })

app_resources.append({
    "name": "database",
    "description": "Lakebase database instance for persistent chat history",
    "database": {
        "database_name": "databricks_postgres",
        "instance_name": lakebase_full_name,
        "permission": "CAN_CONNECT_AND_CREATE"
    }
})

print(f"\nApp resources ({len(app_resources)}):")
for r in app_resources:
    print(f"   - {r['name']}: {r.get('description', '')}")

app_payload = {
    "name": app_full_name,
    "description": "Mag7 Financial Analysis Chatbot powered by Agent Bricks Multi-Agent Supervisor",
    "resources": app_resources
}

# Get auth token for raw requests (needed for long-timeout app creation)
workspace_url = f"https://{host}"
token = notebook_context.apiToken().get()
req_headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

# Check if app exists and create/update
print(f"\nChecking if app '{app_full_name}' exists...")
try:
    existing = w.api_client.do('GET', f'/api/2.0/apps/{app_full_name}')
    print(f"App already exists, updating...")
    result = w.api_client.do('PATCH', f'/api/2.0/apps/{app_full_name}', body=app_payload)
except NotFound:
    print(f"App not found, creating new app (this may take a few minutes)...")
    resp = req.post(
        f"{workspace_url}/api/2.0/apps",
        headers=req_headers,
        json=app_payload,
        timeout=300
    )
    if resp.status_code in (200, 201):
        result = resp.json()
    elif resp.status_code == 504 or 'deadline' in resp.text.lower():
        print("App creation timed out but is still in progress. Waiting...")
        result = None
        for i in range(24):
            time.sleep(5)
            try:
                result = w.api_client.do('GET', f'/api/2.0/apps/{app_full_name}')
                print(f"   App found!")
                break
            except NotFound:
                print(f"   Still waiting... ({(i+1)*5}s)")
        if not result:
            raise Exception(f"App '{app_full_name}' not found after waiting. Check workspace UI.")
    else:
        raise Exception(f"App creation failed: {resp.status_code} {resp.text[:500]}")

app_url = result.get('url', 'N/A')
app_status = result.get('app_status', {}).get('state', 'UNKNOWN')
print(f"\nApp ready: {app_full_name}")
print(f"   Status: {app_status}")
print(f"   URL: {app_url}")

In [0]:
# Step 4b: Grant app service principal access to Genie spaces, UC functions, and catalog/schema

app_full_name = f"{app_name}-{app_resource_suffix}"

# --- Discover the app's service principal (with retry for newly created apps) ---
print(f"Discovering service principal for app '{app_full_name}'...")

sp_id = None
sp_client_id = None
sp_name = None

for attempt in range(12):  # up to 60 seconds
    app_info = w.api_client.do('GET', f'/api/2.0/apps/{app_full_name}')
    sp_id = app_info.get('service_principal_id')
    sp_client_id = app_info.get('service_principal_client_id')
    sp_name = app_info.get('service_principal_name')
    if sp_client_id:
        break
    print(f"   SP not assigned yet, waiting... ({(attempt+1)*5}s)")
    time.sleep(5)

print(f"   Service Principal ID: {sp_id}")
print(f"   Service Principal Client ID: {sp_client_id}")
print(f"   Service Principal Name: {sp_name}")

# Look up the SP's applicationId via SCIM if not in app response
if not sp_client_id and sp_id:
    try:
        sp_data = w.api_client.do('GET', f'/api/2.0/preview/scim/v2/ServicePrincipals/{sp_id}')
        sp_client_id = sp_data.get('applicationId')
        sp_name = sp_data.get('displayName', sp_name)
        print(f"   Resolved applicationId via SCIM: {sp_client_id}")
    except Exception as e:
        print(f"   Warning: Could not look up SP details via SCIM: {e}")

if not sp_client_id:
    print("\nWARNING: Could not determine app service principal applicationId.")
    print("The app may still be provisioning. Grant permissions manually:")
    print(f"  GRANT EXECUTE ON FUNCTION `{catalog}`.`{schema}`.`you_web_search` TO `<sp_client_id>`;")
    print("Or re-run this cell after the app finishes provisioning.")
    # Skip the rest of the cell instead of failing
    raise SystemExit(0)

# --- Ensure UC functions include You.com functions if they exist ---
you_com_function_names = ["you_web_search", "you_content_extract", "you_research"]
print(f"\nChecking for You.com UC functions in {catalog}.{schema}...")
for func_name in you_com_function_names:
    full_path = f"{catalog}.{schema}.{func_name}"
    if full_path not in uc_function_paths:
        try:
            w.functions.get(full_path)
            uc_function_paths.append(full_path)
            print(f"   + Found: {full_path}")
        except Exception:
            pass

vega_path = f"{catalog}.{schema}.generate_vega_lite_spec"
if vega_path not in uc_function_paths:
    try:
        w.functions.get(vega_path)
        uc_function_paths.append(vega_path)
        print(f"   + Found: {vega_path}")
    except Exception:
        pass

print(f"\nAll UC functions to grant EXECUTE on: {uc_function_paths}")

# --- Discover catalogs/schemas ---
catalogs_schemas_to_grant = set()
tables_to_grant_select = set()

if genie_space_ids:
    print(f"\nDiscovering tables from Genie spaces...")
    for space_id in genie_space_ids:
        try:
            space_data = w.api_client.do('GET', f'/api/2.0/data-rooms/{space_id}')
            table_ids = space_data.get('table_identifiers', [])
            print(f"   Genie {space_id} tables: {table_ids}")
            for table_id in table_ids:
                parts = table_id.split('.')
                if len(parts) >= 2:
                    catalogs_schemas_to_grant.add((parts[0], parts[1]))
                    tables_to_grant_select.add(f"{parts[0]}.{parts[1]}")
        except Exception as e:
            print(f"   Warning: Error fetching Genie space {space_id}: {e}")

for func_path in uc_function_paths:
    parts = func_path.split('.')
    if len(parts) == 3:
        catalogs_schemas_to_grant.add((parts[0], parts[1]))

print(f"\n   Catalogs/schemas needing grants: {catalogs_schemas_to_grant}")

# --- Grant Genie space permissions ---
if genie_space_ids:
    print(f"\nGranting Genie space permissions to app SP ({sp_client_id})...")
    for space_id in genie_space_ids:
        try:
            w.api_client.do('PATCH', f'/api/2.0/permissions/genie/{space_id}', body={
                "access_control_list": [{
                    "service_principal_name": sp_client_id,
                    "permission_level": "CAN_RUN"
                }]
            })
            print(f"   Granted CAN_RUN on genie/{space_id}")
        except Exception as e:
            print(f"   Warning: Failed to grant Genie permission: {e}")

# --- Grant USE CATALOG + USE SCHEMA ---
if catalogs_schemas_to_grant:
    print(f"\nGranting catalog/schema access to app SP ({sp_client_id})...")
    catalogs_granted = set()
    for cat, sch in catalogs_schemas_to_grant:
        if cat not in catalogs_granted:
            try:
                spark.sql(f"GRANT USE CATALOG ON CATALOG `{cat}` TO `{sp_client_id}`")
                print(f"   Granted USE CATALOG on {cat}")
                catalogs_granted.add(cat)
            except Exception as e:
                print(f"   Warning: Could not grant USE CATALOG on {cat}: {e}")
        try:
            spark.sql(f"GRANT USE SCHEMA ON SCHEMA `{cat}`.`{sch}` TO `{sp_client_id}`")
            print(f"   Granted USE SCHEMA on {cat}.{sch}")
        except Exception as e:
            print(f"   Warning: Could not grant USE SCHEMA on {cat}.{sch}: {e}")

# --- Grant SELECT on schemas used by Genie spaces ---
if tables_to_grant_select:
    print(f"\nGranting SELECT on data tables for Genie queries...")
    for schema_path in tables_to_grant_select:
        try:
            parts = schema_path.split('.')
            spark.sql(f"GRANT SELECT ON SCHEMA `{parts[0]}`.`{parts[1]}` TO `{sp_client_id}`")
            print(f"   Granted SELECT on {schema_path}")
        except Exception as e:
            print(f"   Warning: Could not grant SELECT on {schema_path}: {e}")

# --- Grant UC function EXECUTE ---
if uc_function_paths:
    print(f"\nGranting UC function EXECUTE to app SP ({sp_client_id})...")
    for func_path in uc_function_paths:
        try:
            parts = func_path.split('.')
            escaped = '`.`'.join(parts)
            spark.sql(f"GRANT EXECUTE ON FUNCTION `{escaped}` TO `{sp_client_id}`")
            print(f"   Granted EXECUTE on {func_path}")
        except Exception as e:
            print(f"   Warning: Could not grant EXECUTE on {func_path}: {e}")

print(f"\nPermission grants complete!")

In [0]:
# Step 5: Deploy source code to the app

# Get current user for the source path
current_user = notebook_context.userName().get()
app_source_path = f"/Workspace/Users/{current_user}/agent-bricks-workshop/chatbot-app"
app_full_name = f"{app_name}-{app_resource_suffix}"

print(f"Deploying source code to app '{app_full_name}'...")
print(f"   Source: {app_source_path}")
print(f"   User: {current_user}")

# Wait for app to be ready to accept deployments
print("   Waiting for app to be ready...")
for attempt in range(12):  # up to 60 seconds
    time.sleep(5)
    app_data = w.api_client.do('GET', f'/api/2.0/apps/{app_full_name}')
    state = app_data.get('compute_status', {}).get('state', '')
    print(f"   App compute state: {state}")
    if state not in ('CREATING', 'STARTING', 'STOPPING'):
        break

# Trigger deployment
print(f"\nTriggering deployment...")
deploy_payload = {"source_code_path": app_source_path}

try:
    deploy_result = w.api_client.do('POST', f'/api/2.0/apps/{app_full_name}/deployments', body=deploy_payload)
    deployment_id = deploy_result.get('deployment_id', 'unknown')
    deploy_status = deploy_result.get('status', {}).get('state', 'STARTING')
    print(f"\nDeployment started!")
    print(f"   Deployment ID: {deployment_id}")
    print(f"   Status: {deploy_status}")
    print(f"   Source: {deploy_result.get('source_code_path', app_source_path)}")
except Exception as e:
    print(f"\nWarning: Deployment issue: {e}")
    print("You can deploy manually: Workspace > Apps > agent-bricks-chatbot-workshop > Deploy")

## 🎉 Chatbot App Deployment Complete!

### What was deployed:
- **Databricks App**: Chatbot UI connected to the Multi-Agent Supervisor
- **Lakebase Database**: Persistent chat history storage
- **Permission Grants**: CAN_QUERY on MAS and all sub-agent endpoints

### Next Steps:
1. Wait for the app to reach RUNNING state
2. Access the app URL in your browser
3. Try asking financial analysis questions!

In [0]:
# Summary
print("\n🎉 Chatbot App Deployment Complete!")
print("\n📋 Summary:")
print(f"  • App Name: {app_full_name}")
print(f"  • MAS Endpoint: {mas_endpoint_name}")
print(f"  • Lakebase Instance: {lakebase_full_name}")
print(f"  • Sub-agent Endpoints: {len(sub_agent_endpoints)}")
for ep in sub_agent_endpoints:
    print(f"    - {ep}")
print(f"\n🚀 Access your app in the Databricks workspace under Apps.")
if app_url and app_url != 'N/A':
    print(f"   URL: {app_url}")